# Project 2 — Lid-Driven Cavity Flow

Incompressible Navier-Stokes solver using:
- **Fractional Step (Projection) Method** (3 sub-steps)
- **Staggered MAC grid** (pressure at cell centers, velocities at faces)
- **Crank-Nicolson** for viscous terms
- **2nd-order Adams-Bashforth** for advection
- **Conjugate Gradient** linear solver (via `scipy`)

## Imports

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.sparse.linalg import cg, LinearOperator

## Parameters

In [7]:
nx = 128
ny = 128
Lx = 1.0
Ly = 1.0
Ustar = 1.0
Re = 100.0
visc = Ustar * Lx / Re
CFL = 0.2

# Domain extent
xL = 0.0
xR = xL + Lx
yB = 0.0
yT = yB + Ly

# Discretization
# NOTE: renamed from 'np' to 'n_cells' to avoid shadowing numpy
n_cells = nx * ny                        # number of pressure DOFs
n_vel   = ny * (nx - 1) + nx * (ny - 1) # number of velocity DOFs
dx = Lx / nx
dy = Ly / ny
dt = CFL * dx
end_time = 10.0

print(f"dt = {dt:.5f}, n_cells = {n_cells}, n_vel = {n_vel}")

dt = 0.00156, n_cells = 16384, n_vel = 32512


## Grid Indexing

Build pointer arrays that map 2D grid indices `(i, j)` to flat 1D array indices.

- `ip[i, j]` — index of pressure at cell center `(i, j)`, `i=0..nx-1`, `j=0..ny-1`
- `iu[i, j]` — index of u-velocity at vertical face between columns `i-1` and `i`, `i=1..nx-1`
- `iv[i, j]` — index of v-velocity at horizontal face between rows `j-1` and `j`, `j=1..ny-1`

Invalid entries (boundaries with no DOF) are set to `-1`.

In [8]:
def func_gen_pointer(nx, ny):
    """
    Generate flat-array index pointers for staggered grid variables.
    Uses 0-based indexing. Invalid (boundary) entries are -1.
    """
    # Pressure: all (nx, ny) cells are DOFs
    ip = np.full((nx, ny), -1, dtype=int)
    count = 0
    for i in range(nx):
        for j in range(ny):
            ip[i, j] = count
            count += 1

    # u-velocity: interior vertical faces, i = 1..nx-1
    iu = np.full((nx + 1, ny), -1, dtype=int)
    count = 0
    for i in range(1, nx):   # i=1..nx-1
        for j in range(ny):
            iu[i, j] = count
            count += 1

    # v-velocity: interior horizontal faces, j = 1..ny-1
    # NOTE: v-DOFs follow u-DOFs in the flat velocity vector, so offset by ny*(nx-1)
    iv = np.full((nx, ny + 1), -1, dtype=int)
    count = ny * (nx - 1)
    for i in range(nx):
        for j in range(1, ny):   # j=1..ny-1
            iv[i, j] = count
            count += 1

    return ip, iu, iv


ip, iu, iv = func_gen_pointer(nx, ny)
print(f"ip range: [{ip.min()}, {ip.max()}]")
print(f"iu DOFs: {(iu >= 0).sum()}  (expected {ny*(nx-1)})")
print(f"iv DOFs: {(iv >= 0).sum()}  (expected {nx*(ny-1)})")

ip range: [0, 16383]
iu DOFs: 16256  (expected 16256)
iv DOFs: 16256  (expected 16256)


## Grid Generation

Physical coordinates of each variable location on the staggered grid.

In [9]:
def func_gen_grid(nx, ny, xL, xR, yB, yT, dx, dy):
    """
    Generate coordinate arrays for each variable type on the staggered grid.

    Returns
    -------
    xp, yp : 1D arrays, pressure cell-center coordinates
    xu, yu : 1D arrays, u-velocity face coordinates (vertical faces, i=1..nx-1)
    xv, yv : 1D arrays, v-velocity face coordinates (horizontal faces, j=1..ny-1)
    """
    # Pressure cell centers
    xp = np.linspace(xL + dx / 2, xR - dx / 2, nx)
    yp = np.linspace(yB + dy / 2, yT - dy / 2, ny)

    # u-velocity: on vertical (x-direction) interior faces
    xu = np.linspace(xL + dx, xR - dx, nx - 1)  # x-coords of interior vertical faces
    yu = yp.copy()                                # same y as pressure centers

    # v-velocity: on horizontal (y-direction) interior faces
    xv = xp.copy()                                # same x as pressure centers
    yv = np.linspace(yB + dy, yT - dy, ny - 1)  # y-coords of interior horizontal faces

    return xp, yp, xu, yu, xv, yv


xp, yp, xu, yu, xv, yv = func_gen_grid(nx, ny, xL, xR, yB, yT, dx, dy)

## Boundary Conditions

Lid-driven cavity: top wall moves at `Ustar`, all others are no-slip.

In [10]:
# u-velocity BCs
uBC_T = Ustar * np.ones(nx)   # top wall moves at Ustar
uBC_B = np.zeros(nx)          # bottom wall: no-slip
uBC_L = np.zeros(ny)          # left wall: no-slip
uBC_R = np.zeros(ny)          # right wall: no-slip

# v-velocity BCs (all zero for lid-driven cavity)
vBC_T = np.zeros(nx)
vBC_B = np.zeros(nx)
vBC_L = np.zeros(ny)
vBC_R = np.zeros(ny)

## Initialization

Start from rest, or load a previous solution from `Results.npz`.

In [ ]:
Init_Soln = True   # set False to attempt loading from Results.npz

if Init_Soln or not os.path.exists('Results.npz'):
    P    = np.zeros(n_cells)
    U    = np.zeros(n_vel)
    time = 0.0
    step = 0
else:
    data = np.load('Results.npz')
    P    = data['P']
    U    = data['U']
    time = float(data['time'])
    step = int(data['step'])
    print(f"Loaded solution at t={time:.4f}, step={step}")

## Discrete Operators

### Divergence `D`
Maps velocity vector (length `n_vel`) → pressure space (length `n_cells`).

### Gradient `G`
Maps pressure vector (length `n_cells`) → velocity space (length `n_vel`). **TODO**

### Laplacian `L`
Applies discrete Laplacian to the velocity vector. **TODO**

### Composite operators
| Symbol | Expression | Role |
|--------|-----------|------|
| `S`    | `I + dt/2·ν·L` | Explicit viscous half-step |
| `R`    | `I - dt/2·ν·L` | Implicit viscous half-step (LHS of Step 1) |
| `T`    | `D·S·G`        | Pressure Poisson operator (LHS of Step 2) |
| `C`    | `S·G`          | Velocity correction (Step 3) |

In [ ]:
def laplace_x(x):
    """
    Apply discrete Laplacian to velocity vector x (length n_vel).
    Must handle both u and v DOFs on the staggered grid.
    TODO: implement using central-difference stencil on staggered grid.
    """
    raise NotImplementedError("laplace_x not yet implemented")


def bc_laplace():
    """
    Boundary contribution to the Laplacian RHS (velocity space).
    Encodes uBC_T, uBC_B, uBC_L, uBC_R and vBC_* into a vector of length n_vel.
    TODO: implement.
    """
    raise NotImplementedError("bc_laplace not yet implemented")


def bc_div():
    """
    Boundary contribution to the divergence RHS (pressure space).
    Arises from boundary face fluxes excluded from Opt_D interior loop.
    TODO: implement.
    """
    raise NotImplementedError("bc_div not yet implemented")


def opt_D(x):
    """
    Divergence operator D: velocity (n_vel,) -> pressure (n_cells,).
    NOTE: currently only fills interior cells; boundary cells are missing.
    TODO: add boundary-adjacent cells using BC face velocities.
    """
    sol = np.zeros(n_cells)
    for j in range(1, ny - 1):
        for i in range(1, nx - 1):
            sol[ip[i, j]] = (x[iu[i + 1, j]] - x[iu[i, j]]) / dx + \
                             (x[iv[i, j + 1]] - x[iv[i, j]]) / dy
    return sol


def opt_G(x):
    """
    Gradient operator G: pressure (n_cells,) -> velocity (n_vel,).
    TODO: implement central-difference gradient on staggered grid.
    """
    grad_p = np.zeros(n_vel)
    # TODO: loop over interior u-faces and v-faces
    #   For u-face at (i, j): grad_p[iu[i,j]] = (x[ip[i,j]] - x[ip[i-1,j]]) / dx
    #   For v-face at (i, j): grad_p[iv[i,j]] = (x[ip[i,j]] - x[ip[i,j-1]]) / dy
    raise NotImplementedError("opt_G not yet implemented")
    return grad_p


def opt_S(x):
    """Explicit viscous half-step: (I + dt/2 * visc * L) x"""
    return x + (dt / 2.0 * visc) * laplace_x(x)


def opt_R(x):
    """Implicit viscous half-step: (I - dt/2 * visc * L) x  [LHS of Step 1]"""
    return x - (dt / 2.0 * visc) * laplace_x(x)


def opt_T(x):
    """Pressure Poisson operator: D * S * G  [LHS of Step 2]"""
    return opt_D(opt_S(opt_G(x)))


def opt_C(x):
    """Velocity correction operator: S * G  [used in Step 3]"""
    return opt_S(opt_G(x))

In [ ]:
import operators as ops


## Time Loop — Fractional Step Method

**Step 1** — Solve for intermediate velocity `U*`:
$$
\underbrace{(I - \tfrac{\Delta t}{2}\nu L)}_{R} U^* =
\underbrace{(I + \tfrac{\Delta t}{2}\nu L)}_{S} U^n
+ \Delta t \left(\tfrac{3}{2} N^n - \tfrac{1}{2} N^{n-1}\right)
+ \Delta t\, \nu\, \text{BC}_L
$$

**Step 2** — Solve pressure Poisson equation:
$$
\underbrace{D\,S\,G}_{T}\, P = \frac{1}{\Delta t}\left(D\,U^* + \text{BC}_D\right)
$$

**Step 3** — Correct velocity to enforce $\nabla\cdot U = 0$:
$$
U^{n+1} = U^* - \Delta t\, C\, P
$$

In [ ]:
adv_prev = np.zeros(n_vel)   # N(U) at step n-1 for Adams-Bashforth startup

while time < end_time:
    step += 1
    time += dt   # BUG FIX: original MATLAB had 'time = time + 1' (should be + dt)

    adv0 = adv(U)       # N(U^n)
    adv1 = adv_prev     # N(U^{n-1})

    # --- Step 1: intermediate velocity ---
    RHS_U = opt_S(U) + dt * (1.5 * adv0 - 0.5 * adv1) + dt * visc * bc_laplace()
    UF = cg_solve(opt_R, RHS_U, tol=1e-8)

    # --- Step 2: pressure ---
    RHS_P = opt_D(UF) / dt + bc_div() / dt
    P = cg_solve(opt_T, RHS_P, tol=1e-8)

    # --- Step 3: velocity correction ---
    U = UF - dt * opt_C(P)

    adv_prev = adv0   # store for next step (Adams-Bashforth)

    if step % 100 == 0:
        print(f"step={step:5d}, time={time:.4f}")

# Save results
np.savez('Results.npz', P=P, U=U, time=time, step=step)

## Post-Processing / Visualization

TODO: reconstruct 2D velocity fields from the flat `U` vector and plot.

In [ ]:
# TODO: reconstruct 2D u and v fields from U vector using iu/iv pointers
# Example skeleton:
#
# u_field = np.zeros((nx - 1, ny))
# for i in range(1, nx):
#     for j in range(ny):
#         u_field[i - 1, j] = U[iu[i, j]]
#
# v_field = np.zeros((nx, ny - 1))
# for i in range(nx):
#     for j in range(1, ny):
#         v_field[i, j - 1] = U[iv[i, j]]
#
# p_field = P.reshape((nx, ny))
#
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# axes[0].contourf(xu, yu, u_field.T); axes[0].set_title('u-velocity')
# axes[1].contourf(xv, yv, v_field.T); axes[1].set_title('v-velocity')
# axes[2].contourf(xp, yp, p_field.T); axes[2].set_title('pressure')
# plt.tight_layout()
# plt.show()
pass